In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import json
import glob
import os

In [2]:
# 📁 Папка, где лежат parquet-чанки
CHUNKS_DIR = r"C:\Проекты\project_ml_monopoly\notebooks\ai_продукты"

def load_all_chunks_to_one_df(chunks_dir: str) -> pd.DataFrame:
    # Смотрим все файлы си эс ви в папке
    pattern = os.path.join(chunks_dir, "*.csv")
    files = sorted(glob.glob(pattern))
    # Заглушка
    if not files:
        raise FileNotFoundError(f"Не найдено ни одного файла по маске: {pattern}")

    print(f"Найдено csv-файлов: {len(files)}")
    # Складываем все из них в датасет
    dfs = []
    for file in files:
        df_chunk = pd.read_csv(file)
        dfs.append(df_chunk)
    # Складываем все из них в датасет
    df_full = pd.concat(dfs, ignore_index=True)

    print(f"Итоговый размер: {df_full.shape[0]:,} строк × {df_full.shape[1]} колонок")
    mem_mb = df_full.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"Память: ~{mem_mb:.2f} MB")

    return df_full


if __name__ == "__main__":
    df_product = load_all_chunks_to_one_df(CHUNKS_DIR)


Найдено csv-файлов: 13


C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\424623191.py:16: DtypeWarning: Columns (65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_chunk = pd.read_csv(file)
C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\424623191.py:16: DtypeWarning: Columns (65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_chunk = pd.read_csv(file)
C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\424623191.py:16: DtypeWarning: Columns (65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_chunk = pd.read_csv(file)
C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\424623191.py:16: DtypeWarning: Columns (65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_chunk = pd.read_csv(file)
C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\424623191.py:16: DtypeWarning: Columns (65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_chunk = pd.

Итоговый размер: 742,222 строк × 76 колонок
Память: ~3006.19 MB


In [3]:
df_product = df_product.drop_duplicates()

In [4]:
df_cut = df_product[['Код товара',
                     'МНН',
                     'Форма выпуска',
                     'Лекарственная форма',
                     'Группа',
                     'Препарат предметно-количественного учета',
                     'Сезонность 1',
                     'Условия хранения',
                     'Рейтинг внешний',
                     'ЖНВЛС',
                     'ЭО общая',
                     'Статус',
                     'Обязательный ассортимент']]

In [5]:

def load_saved_model(model_dir, model_filename):
    """Загружает модель и все связанные файлы."""
    
    model_path = os.path.join(model_dir, f"{model_filename}.cbm")
    metadata_path = os.path.join(model_dir, f"{model_filename}_metadata.json")
    threshold_path = os.path.join(model_dir, f"{model_filename}_thresholds.csv")
    features_path = os.path.join(model_dir, f"{model_filename}_features.json")
    
    # Модель
    model = CatBoostClassifier()
    model.load_model(model_path)
    
    # Метаданные
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    # Thresholds
    threshold_df = pd.read_csv(threshold_path)
    
    # Features
    with open(features_path, 'r', encoding='utf-8') as f:
        features_info = json.load(f)
    
    print(f"✅ Модель загружена: {model_path}")
    print(f"   Фичей: {len(features_info['features'])}")
    print(f"   PR-AUC: {metadata['metrics']['pr_auc']:.4f}")
    
    return {
        'model': model,
        'features': features_info['features'],
        'cat_features': features_info['cat_features'],
        'metadata': metadata,
        'threshold_df': threshold_df,
    }

In [6]:
model_data = load_saved_model(
    model_dir="models",
    model_filename="catboost_oos_20260224_111000"  # ← твоё имя файла
)

✅ Модель загружена: models\catboost_oos_20260224_111000.cbm
   Фичей: 156
   PR-AUC: 0.6329


# подготовка данных

In [7]:
# 📁 Папка, где лежат parquet-чанки
CHUNKS_DIR = r"C:\Проекты\project_ml_monopoly\notebooks\ai_stock_4years_chunks"

def load_all_chunks_to_one_df(chunks_dir: str) -> pd.DataFrame:
    pattern = os.path.join(chunks_dir, "*.csv")
    files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(f"Не найдено ни одного файла по маске: {pattern}")

    print(f"Найдено csv-файлов: {len(files)}")

    dfs = []
    for file in files:
        df_chunk = pd.read_csv(file)
        dfs.append(df_chunk)

    df_full = pd.concat(dfs, ignore_index=True)

    print(f"Итоговый размер: {df_full.shape[0]:,} строк × {df_full.shape[1]} колонок")
    mem_mb = df_full.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"Память: ~{mem_mb:.2f} MB")

    return df_full


if __name__ == "__main__":
    df_all = load_all_chunks_to_one_df(CHUNKS_DIR)


Найдено csv-файлов: 4
Итоговый размер: 1,143,322 строк × 19 колонок
Память: ~781.30 MB


In [8]:
df_all = df_all.drop(['Остаток общий', 'Количество Ирбит', 'Остаток в пути'], axis =1)

In [9]:
df_all['Дата'].max()

'2026-03-03'

In [10]:
df_all = df_all[df_all['Активный КАГ'] == 'Да']

In [11]:
df_all = pd.merge(df_all, df_cut, how='left', on='Код товара')

In [12]:
df_all = df_all.drop(['Цена пульса', 'Цена катрена', 'Цена протека', 'Цена фармкомплекта', 'Активный КАГ'], axis = True)

In [13]:
df_all = df_all.drop_duplicates()

In [14]:
df_all.columns = [
    'date', 
    'cag_id',          # Added this to match column index 1
    'code_kag', 
    'name_kag', 
    'code_product', 
    'name_product', 
    'gk_stock', 
    'puls_stock', 
    'katren_stock', 
    'protek_stock', 
    'farm_stock',
    'МНН',
    'Форма выпуска',
    'Лекарственная форма',
    'Группа',
    'Препарат предметно-количественного учета',
    'Сезонность 1',
    'Условия хранения',
    'Рейтинг внешний',
    'ЖНВЛС',
    'ЭО общая',
    'Статус',
    'Обязательный ассортимент'
]

In [15]:
df_all['date'] = pd.to_datetime(df_all['date'])
df_all['code_kag'] = df_all['code_kag'].astype(str)
df_sum = df_all.groupby(['date', 'code_kag'], sort=False)[['puls_stock', 'gk_stock', 'katren_stock', 'protek_stock', 'farm_stock']].sum()
masked = df_all[['date', 'code_kag'] + ['puls_stock', 'gk_stock', 'katren_stock', 'protek_stock', 'farm_stock']].copy()
df_max = df_all.groupby(['date', 'code_kag'], sort=False)[['puls_stock', 'gk_stock', 'katren_stock', 'protek_stock', 'farm_stock']].max()
df_min = df_all.groupby(['date', 'code_kag'], sort=False)[['puls_stock', 'gk_stock', 'katren_stock', 'protek_stock', 'farm_stock']].min()
idx = df_sum.index
df_max = df_max.reindex(idx)
df_min = df_min.reindex(idx)
is_dedup = df_max.eq(df_min) | df_max.isna()
final_df = df_sum.mask(is_dedup, df_max.fillna(0))
final_df = final_df.reset_index()
final_df['code_kag'] = final_df['code_kag'].astype(int)

In [16]:
import pandas as pd
import holidays
def filter_working_days(df, date_col='date', country='RU'):
    """
    Исключительно удаляет выходные (Сб, Вс) и государственные праздники.
    """
    # 1. Приводим к формату даты
    df[date_col] = pd.to_datetime(df[date_col])
    
    # 2. Получаем список праздников для нужной страны и годов
    years = df[date_col].dt.year.unique()
    public_holidays = holidays.CountryHoliday(country, years=years)
    
    # 3. Фильтруем: 
    # - день недели < 5 (Пн-Пт)
    # - даты нет в списке государственных праздников
    is_not_weekend = df[date_col].dt.dayofweek < 5
    is_not_holiday = ~df[date_col].isin(public_holidays)
    
    return df[is_not_weekend & is_not_holiday].copy()


final_df = filter_working_days(final_df)

C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\1574789871.py:18: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  is_not_holiday = ~df[date_col].isin(public_holidays)


In [17]:
df_all['code_kag'] = df_all['code_kag'].astype(int)

In [18]:
ref_columns = ['code_kag', 'МНН', 'Форма выпуска', 'Лекарственная форма',
               'Группа', 'Препарат предметно-количественного учета',
               'Сезонность 1', 'Условия хранения', 'Рейтинг внешний',
               'ЖНВЛС', 'ЭО общая', 'Статус', 'Обязательный ассортимент']

ref_df = df_all[ref_columns].drop_duplicates(subset='code_kag')

final_df = pd.merge(final_df, ref_df, how='left', on='code_kag')

In [19]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410581 entries, 0 to 410580
Data columns (total 19 columns):
 #   Column                                    Non-Null Count   Dtype         
---  ------                                    --------------   -----         
 0   date                                      410581 non-null  datetime64[ns]
 1   code_kag                                  410581 non-null  int64         
 2   puls_stock                                410581 non-null  int64         
 3   gk_stock                                  410581 non-null  int64         
 4   katren_stock                              410581 non-null  int64         
 5   protek_stock                              410581 non-null  int64         
 6   farm_stock                                410581 non-null  int64         
 7   МНН                                       309606 non-null  object        
 8   Форма выпуска                             309802 non-null  object        
 9   Лекарственная ф

In [20]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import json
import os


# ================================================================
# ФУНКЦИИ СОЗДАНИЯ ФИЧЕЙ (те же что при обучении)
# ================================================================

def create_all_features_clean(df, date_col='date', product_col='code_kag', 
                               category_col='NM_F', mnn_col=None):
    """
    ЕДИНЫЙ PIPELINE без leakage и без абсолютных величин.
    Все признаки — относительные или безразмерные.
    """
    df = df.sort_values([product_col, date_col]).copy()
    
    stock_cols = ['puls_stock', 'katren_stock', 'protek_stock', 'farm_stock']
    
    if 'gk_stock' in df.columns:
        stock_cols = stock_cols + ['gk_stock']
    
    dist_names = [c.replace('_stock', '') for c in stock_cols]
    grouped = df.groupby(product_col)
    
    # 0. БАЗОВЫЕ АГРЕГАТЫ
    df['total_stock'] = df[stock_cols].sum(axis=1)
    df['full_oos'] = (df['total_stock'] == 0).astype(int)
    
    df['_product_median_exp'] = grouped['total_stock'].transform(
        lambda x: x.expanding(min_periods=7).median()
    )
    df['_product_max_exp'] = grouped['total_stock'].transform(
        lambda x: x.expanding(min_periods=7).max()
    )
    df['_norm_base'] = df['_product_median_exp'].fillna(df['total_stock'].replace(0, 1))
    
    # 1. ДЕЛЬТЫ
    for col in stock_cols:
        prev_val = grouped[col].shift(1)
        df[f'{col}_diff_pct'] = (df[col] - prev_val) / (prev_val + 1)
    
    df['total_stock_diff_pct'] = grouped['total_stock'].pct_change().clip(-1, 10)
    
    # 2. ПРОДАЖИ/ПОСТАВКИ
    for col in stock_cols:
        prefix = col.replace('_stock', '')
        diff_pct = df[f'{col}_diff_pct']
        df[f'{prefix}_sales_pct'] = diff_pct.clip(upper=0).abs()
        df[f'{prefix}_supply_pct'] = diff_pct.clip(lower=0)
    
    sales_pct_cols = [f'{d}_sales_pct' for d in dist_names]
    supply_pct_cols = [f'{d}_supply_pct' for d in dist_names]
    
    df['avg_sales_pct'] = df[sales_pct_cols].mean(axis=1)
    df['avg_supply_pct'] = df[supply_pct_cols].mean(axis=1)
    df['max_sales_pct'] = df[sales_pct_cols].max(axis=1)
    
    # 3. СКОЛЬЗЯЩИЕ
    for window in [7, 14, 30]:
        df[f'avg_sales_pct_{window}d'] = grouped['avg_sales_pct'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
        df[f'sales_cv_{window}d'] = grouped['avg_sales_pct'].transform(
            lambda x: x.rolling(window, min_periods=3).std() / 
                      (x.rolling(window, min_periods=3).mean() + 0.001)
        ).clip(0, 10)
    
    # 4. DAYS OF STOCK
    df['days_of_stock'] = np.where(
        df['avg_sales_pct_7d'] > 0.001,
        1 / df['avg_sales_pct_7d'],
        999
    ).clip(0, 365)
    
    df['days_of_stock_cat'] = pd.cut(
        df['days_of_stock'],
        bins=[0, 3, 7, 14, 30, 365, 1000],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype(float)
    
    # 5. ОТНОСИТЕЛЬНЫЕ УРОВНИ ЗАПАСА
    df['stock_vs_median'] = df['total_stock'] / (df['_norm_base'] + 1)
    df['stock_vs_max'] = df['total_stock'] / (df['_product_max_exp'] + 1)
    
    for lag in [7, 14]:
        lag_stock = grouped['total_stock'].shift(lag)
        df[f'stock_change_{lag}d_pct'] = (
            (df['total_stock'] - lag_stock) / (lag_stock + 1)
        ).clip(-1, 5)
    
    # 6. ДОЛИ ДИСТРИБЬЮТОРОВ
    for col in stock_cols:
        df[f'{col}_share'] = df[col] / (df['total_stock'] + 1)
    
    share_cols = [f'{col}_share' for col in stock_cols]
    df['stock_hhi'] = sum(df[c] ** 2 for c in share_cols)
    
    df['stock_cv_cross'] = (
        df[stock_cols].std(axis=1) / (df[stock_cols].mean(axis=1) + 1)
    ).clip(0, 10)
    
    # 7. ФЛАГИ РИСКА
    df['n_zero'] = (df[stock_cols] == 0).sum(axis=1)
    
    for col in stock_cols:
        prefix = col.replace('_stock', '')
        col_median_exp = grouped[col].transform(
            lambda x: x.expanding(min_periods=7).median()
        ).fillna(df[col])
        df[f'{prefix}_is_low'] = (df[col] < col_median_exp * 0.1).astype(int)
        df[f'{prefix}_is_critical'] = (df[col] < col_median_exp * 0.05).astype(int)
    
    low_cols = [f'{d}_is_low' for d in dist_names]
    critical_cols = [f'{d}_is_critical' for d in dist_names]
    df['n_low'] = df[low_cols].sum(axis=1)
    df['n_critical'] = df[critical_cols].sum(axis=1)
    
    # 8. СИНХРОННОСТЬ
    for col in stock_cols:
        prefix = col.replace('_stock', '')
        df[f'{prefix}_falling'] = (df[f'{col}_diff_pct'] < -0.01).astype(int)
        df[f'{prefix}_rising'] = (df[f'{col}_diff_pct'] > 0.01).astype(int)
    
    falling_cols = [f'{d}_falling' for d in dist_names]
    rising_cols = [f'{d}_rising' for d in dist_names]
    df['n_dist_falling'] = df[falling_cols].sum(axis=1)
    df['n_dist_rising'] = df[rising_cols].sum(axis=1)
    df['all_falling'] = (df['n_dist_falling'] == len(dist_names)).astype(int)
    
    # 9. ПОСТАВКИ
    df['had_supply'] = (df['avg_supply_pct'] > 0.005).astype(int)
    df['days_since_supply'] = grouped['had_supply'].transform(
        lambda x: x.groupby((x == 1).cumsum()).cumcount()
    )
    
    df['avg_supply_interval_30d'] = grouped['days_since_supply'].transform(
        lambda x: x.rolling(30, min_periods=7).mean()
    )
    df['supply_interval_ratio'] = (
        df['days_since_supply'] / (df['avg_supply_interval_30d'] + 1)
    ).clip(0, 5)
    df['supply_overdue'] = (df['supply_interval_ratio'] > 1.5).astype(int)
    
    df['supply_freq_30d'] = grouped['had_supply'].transform(
        lambda x: x.rolling(30, min_periods=1).mean()
    )
    
    # 10. ТРЕНДЫ
    df['sales_trend'] = (
        df['avg_sales_pct_7d'] / (df['avg_sales_pct_30d'] + 0.001)
    ).clip(0.1, 10)
    
    df['stock_trend_7d'] = grouped['total_stock'].transform(
        lambda x: x.pct_change(7)
    ).clip(-1, 1)
    
    df['stock_trend_14d'] = grouped['total_stock'].transform(
        lambda x: x.pct_change(14)
    ).clip(-1, 1)
    
    # 11. Z-SCORES
    for col_name, source in [('stock', 'stock_vs_median'), ('sales', 'avg_sales_pct')]:
        rolling_mean = grouped[source].transform(lambda x: x.rolling(30, min_periods=7).mean())
        rolling_std = grouped[source].transform(lambda x: x.rolling(30, min_periods=7).std())
        df[f'{col_name}_zscore'] = (
            (df[source] - rolling_mean) / (rolling_std + 0.001)
        ).clip(-5, 5)
    
    df['stock_anomaly_low'] = (df['stock_zscore'] < -2).astype(int)
    df['sales_spike'] = (df['sales_zscore'] > 2).astype(int)
    
    # 12. СОБЫТИЯ ПЕРЕХОДА В НОЛЬ
    for col in stock_cols:
        prefix = col.replace('_stock', '')
        is_zero = (df[col] == 0).astype(int)
        was_positive = grouped[col].shift(1) > 0
        df[f'{prefix}_went_zero'] = (is_zero & was_positive).astype(int)
    
    went_zero_cols = [f'{d}_went_zero' for d in dist_names]
    df['n_went_zero_today'] = df[went_zero_cols].sum(axis=1)
    
    # 13. MOMENTUM / STREAK
    def consecutive_count(series):
        result = np.zeros(len(series), dtype=np.int32)
        count = 0
        for i, val in enumerate(series.values):
            if val:
                count += 1
            else:
                count = 0
            result[i] = count
        return result
    
    df['consecutive_falls'] = grouped['all_falling'].transform(consecutive_count)
    
    df['no_supply_anywhere'] = (df['avg_supply_pct'] < 0.001).astype(int)
    df['consecutive_no_supply'] = grouped['no_supply_anywhere'].transform(consecutive_count)
    
    df['n_zero_increasing'] = (df['n_zero'] > grouped['n_zero'].shift(1)).astype(int)
    df['n_zero_increase_streak'] = grouped['n_zero_increasing'].transform(consecutive_count)
    
    # 14. ИСТОРИЯ OOS
    for window in [7, 30]:
        df[f'oos_count_{window}d'] = grouped['full_oos'].transform(
            lambda x: x.rolling(window, min_periods=1).sum()
        )
    
    df['had_oos_7d'] = (df['oos_count_7d'] > 0).astype(int)
    df['had_oos_30d'] = (df['oos_count_30d'] > 0).astype(int)
    df['oos_rate_30d'] = df['oos_count_30d'] / 30
    
    df['days_since_oos'] = grouped['full_oos'].transform(
        lambda x: x.groupby((x == 1).cumsum()).cumcount()
    )
    
    # 15. ПРЕДСТАВЛЕННОСТЬ
    for col in stock_cols:
        prefix = col.replace('_stock', '')
        df[f'{prefix}_was_active'] = grouped[col].transform(
            lambda x: (x.shift(1).expanding().max() > 0).astype(int)
        )
        df[f'{prefix}_is_active_now'] = (df[col] > 0).astype(int)
    
    # 16. КАТЕГОРИЙНЫЙ КОНТЕКСТ
    if category_col and category_col in df.columns:
        df['cat_oos_rate_today'] = df.groupby([category_col, date_col])['full_oos'].transform('mean')
        df['cat_oos_rate_7d'] = df.groupby(category_col)['full_oos'].transform(
            lambda x: x.rolling(7, min_periods=1).mean()
        )
        df['category_stress'] = (df['cat_oos_rate_7d'] > 0.1).astype(int)
    
    # 17. MNN КОНТЕКСТ
    if mnn_col and mnn_col in df.columns:
        mnn_stats = df.groupby([mnn_col, date_col]).agg({
            'total_stock': 'sum',
            'full_oos': 'mean'
        }).reset_index()
        mnn_stats.columns = [mnn_col, date_col, '_mnn_total_stock', 'mnn_oos_rate']
        
        df = df.merge(mnn_stats, on=[mnn_col, date_col], how='left')
        df['sku_share_in_mnn'] = df['total_stock'] / (df['_mnn_total_stock'] + 1)
        df = df.drop(columns=['_mnn_total_stock'], errors='ignore')
    
    # 18. КАЛЕНДАРНЫЕ
    df['dayofweek'] = pd.to_datetime(df[date_col]).dt.dayofweek
    df['month'] = pd.to_datetime(df[date_col]).dt.month
    
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    # CLEANUP
    internal_cols = [c for c in df.columns if c.startswith('_')]
    df = df.drop(columns=internal_cols, errors='ignore')
    df = df.replace([np.inf, -np.inf], np.nan)
    
    return df


def create_predictive_features(df, date_col='date', product_col='code_kag',
                                category_col='NM_F', mnn_col=None):
    """
    Дополнительные предиктивные фичи для early warning.
    """
    df = df.sort_values([product_col, date_col]).copy()
    
    stock_cols = ['puls_stock', 'katren_stock', 'protek_stock', 'farm_stock']
    if 'gk_stock' in df.columns:
        stock_cols.append('gk_stock')
    dist_names = [c.replace('_stock', '') for c in stock_cols]
    
    grouped = df.groupby(product_col)
    
    # 1. VELOCITY
    df['sales_velocity'] = (
        df['avg_sales_pct_7d'] / (df['avg_sales_pct_14d'] + 0.001)
    ).clip(0, 5)
    
    df['sales_accelerating'] = (df['sales_velocity'] > 1.3).astype(int)
    
    df['sales_acceleration'] = grouped['avg_sales_pct_7d'].diff()
    df['sales_acceleration_norm'] = (
        df['sales_acceleration'] / (df['avg_sales_pct_14d'] + 0.001)
    ).clip(-5, 5)
    
    # 2. RUNWAY
    df['runway_critical'] = (df['days_of_stock'] < 7).astype(int)
    df['runway_warning'] = (df['days_of_stock'] < 14).astype(int)
    df['runway_caution'] = (df['days_of_stock'] < 30).astype(int)
    
    for col in stock_cols:
        prefix = col.replace('_stock', '')
        df[f'{prefix}_runway'] = np.where(
            df[f'{prefix}_sales_pct'] > 0.001,
            df[f'{col}_share'] / df[f'{prefix}_sales_pct'],
            999
        ).clip(0, 365)
    
    runway_cols = [f'{d}_runway' for d in dist_names]
    df['min_dist_runway'] = df[runway_cols].min(axis=1)
    df['n_dist_runway_critical'] = (df[runway_cols] < 7).sum(axis=1)
    
    # 3. FIRST MOVER
    for col in stock_cols:
        prefix = col.replace('_stock', '')
        df[f'{prefix}_went_zero'] = (
            (df[col] == 0) & (grouped[col].shift(1) > 0)
        ).astype(int)
    
    for col in stock_cols:
        prefix = col.replace('_stock', '')
        df[f'{prefix}_first_zero_rate_60d'] = grouped[f'{prefix}_went_zero'].transform(
            lambda x: x.rolling(60, min_periods=7).mean()
        )
    
    # 4. КАСКАДНЫЙ ЭФФЕКТ
    df['any_zero_today'] = (df['n_zero'] > 0).astype(int)
    df['n_zero_change'] = grouped['n_zero'].diff()
    df['zeros_spreading'] = (df['n_zero_change'] > 0).astype(int)
    
    df['zeros_spread_streak'] = grouped['zeros_spreading'].transform(
        lambda x: x.groupby((x != x.shift()).cumsum()).cumcount() + 1
    ) * df['zeros_spreading']
    
    # 5. SUPPLY STRESS
    df['supply_sales_ratio'] = (
        df['avg_supply_pct'] / (df['avg_sales_pct'] + 0.001)
    ).clip(0, 10)
    
    df['supply_deficit'] = (df['supply_sales_ratio'] < 0.8).astype(int)
    
    df['supply_deficit_14d'] = grouped['supply_deficit'].transform(
        lambda x: x.rolling(14, min_periods=1).sum()
    )
    
    df['net_flow'] = df['avg_supply_pct'] - df['avg_sales_pct']
    df['net_flow_7d'] = grouped['net_flow'].transform(
        lambda x: x.rolling(7, min_periods=1).mean()
    )
    df['persistent_outflow'] = (df['net_flow_7d'] < -0.01).astype(int)
    
    # 6. MARKET CONTAGION
    if category_col and category_col in df.columns:
        df['cat_n_oos_today'] = df.groupby([category_col, date_col])['full_oos'].transform('sum')
        df['cat_n_products'] = df.groupby([category_col, date_col])[product_col].transform('nunique')
        df['cat_oos_ratio'] = df['cat_n_oos_today'] / (df['cat_n_products'] + 1)
        df['cat_oos_growth'] = df.groupby(category_col)['cat_oos_ratio'].diff()
        df['category_deteriorating'] = (df['cat_oos_growth'] > 0.02).astype(int)
    
    if mnn_col and mnn_col in df.columns:
        df['mnn_oos_rate_today'] = df.groupby([mnn_col, date_col])['full_oos'].transform('mean')
        df['mnn_n_oos'] = df.groupby([mnn_col, date_col])['full_oos'].transform('sum')
        df['mnn_stress'] = (df['mnn_oos_rate_today'] > 0.2).astype(int)
    
    # 7. ИСТОРИЧЕСКАЯ УЯЗВИМОСТЬ
    df['oos_frequency_90d'] = grouped['full_oos'].transform(
        lambda x: x.rolling(90, min_periods=14).mean()
    )
    
    def avg_oos_duration(series):
        result = np.zeros(len(series))
        durations = []
        current_duration = 0
        for i, val in enumerate(series.values):
            if val == 1:
                current_duration += 1
            else:
                if current_duration > 0:
                    durations.append(current_duration)
                current_duration = 0
            if len(durations) > 0:
                result[i] = np.mean(durations[-10:])
        return result
    
    df['avg_oos_duration'] = grouped['full_oos'].transform(avg_oos_duration)
    
    df['chronic_oos'] = (
        (df['oos_frequency_90d'] > 0.1) | (df['avg_oos_duration'] > 3)
    ).astype(int)
    
    # 8. СЕЗОННОСТЬ
    df['day_of_month'] = pd.to_datetime(df[date_col]).dt.day
    df['is_month_start'] = (df['day_of_month'] <= 5).astype(int)
    df['is_month_end'] = (df['day_of_month'] >= 25).astype(int)
    
    df['quarter'] = pd.to_datetime(df[date_col]).dt.quarter
    df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
    df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)
    
    # 9. КОМБИНИРОВАННЫЕ СИГНАЛЫ
    red_flags = [
        'runway_critical', 'sales_accelerating', 'supply_deficit',
        'zeros_spreading', 'persistent_outflow', 'stock_anomaly_low'
    ]
    existing_flags = [f for f in red_flags if f in df.columns]
    df['n_red_flags'] = df[existing_flags].sum(axis=1)
    
    df['critical_combination'] = (
        (df['days_of_stock'] < 14) &
        (df['days_since_supply'] > 7) &
        (df['sales_velocity'] > 1.2)
    ).astype(int)
    
    return df


# ================================================================
# ФУНКЦИИ ДЛЯ INFERENCE
# ================================================================

def load_saved_model(model_dir, model_filename):
    """Загружает модель и все связанные файлы."""
    
    model_path = os.path.join(model_dir, f"{model_filename}.cbm")
    metadata_path = os.path.join(model_dir, f"{model_filename}_metadata.json")
    threshold_path = os.path.join(model_dir, f"{model_filename}_thresholds.csv")
    features_path = os.path.join(model_dir, f"{model_filename}_features.json")
    
    # Модель
    model = CatBoostClassifier()
    model.load_model(model_path)
    print(f"✅ Модель загружена: {model_path}")
    
    # Метаданные
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    # Thresholds
    threshold_df = pd.read_csv(threshold_path)
    
    # Features
    with open(features_path, 'r', encoding='utf-8') as f:
        features_info = json.load(f)
    
    print(f"   Фичей: {len(features_info['features'])}")
    print(f"   PR-AUC: {metadata['metrics']['pr_auc']:.4f}")
    
    return {
        'model': model,
        'features': features_info['features'],
        'cat_features': features_info.get('cat_features', []),
        'metadata': metadata,
        'threshold_df': threshold_df,
    }


def prepare_data_for_inference(df, date_col='date', product_col='code_kag',
                                category_col='NM_F', mnn_col=None):
    """
    Подготовка данных для ПРЕДСКАЗАНИЯ.
    
    ВАЖНО: 
    - НЕ обрезает данные (в отличие от обучения)
    - НЕ создаёт таргет
    - Нужны данные за 30+ дней для корректных rolling фичей
    """
    
    print("="*60)
    print("📊 ПОДГОТОВКА ДАННЫХ ДЛЯ INFERENCE")
    print("="*60)
    
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    
    min_date = df[date_col].min()
    max_date = df[date_col].max()
    n_days = (max_date - min_date).days + 1
    
    print(f"\n📅 Период: {min_date.date()} — {max_date.date()} ({n_days} дней)")
    print(f"   Продуктов: {df[product_col].nunique():,}")
    print(f"   Строк: {len(df):,}")
    
    if n_days < 30:
        print(f"⚠️ ВНИМАНИЕ: Рекомендуется минимум 30 дней для корректных rolling фичей!")
    
    # Шаг 1: Базовые фичи
    print("\n🔧 Шаг 1: Создание базовых фичей...")
    df = create_all_features_clean(
        df, 
        date_col=date_col,
        product_col=product_col,
        category_col=category_col, 
        mnn_col=mnn_col
    )
    print(f"   ✅ Колонок: {len(df.columns)}")
    
    # Шаг 2: Предиктивные фичи
    print("\n🔧 Шаг 2: Создание предиктивных фичей...")
    df = create_predictive_features(
        df, 
        date_col=date_col,
        product_col=product_col,
        category_col=category_col, 
        mnn_col=mnn_col
    )
    print(f"   ✅ Колонок: {len(df.columns)}")
    
    # НЕ вызываем create_simple_oos_target — он обрезает данные!
    # НЕ вызываем dropna — нам нужен последний день!
    
    print(f"\n✅ Готово: {len(df):,} строк, {len(df.columns)} колонок")
    
    return df


def predict_oos_risk(df, model_data, threshold=None, top_k=None):
    """
    Предсказание риска OOS.
    """
    
    model = model_data['model']
    features = model_data['features']
    
    # Копия для добавления недостающих фичей
    df_pred = df.copy()
    
    # Проверяем фичи
    available = [f for f in features if f in df_pred.columns]
    missing = [f for f in features if f not in df_pred.columns]
    
    if missing:
        print(f"⚠️ Отсутствуют {len(missing)} фичей — добавляем как -999:")
        print(f"   {missing}")
        for f in missing:
            df_pred[f] = -999
    
    # Берём ВСЕ фичи В ПРАВИЛЬНОМ ПОРЯДКЕ
    X = df_pred[features].fillna(-999)
    
    # Предсказание
    risk_scores = model.predict_proba(X)[:, 1]
    
    # Результат
    result = df.copy()
    result['risk_score'] = risk_scores
    result['risk_rank'] = result['risk_score'].rank(ascending=False).astype(int)
    
    # Threshold
    if threshold is None:
        threshold = model_data['metadata']['recommended_thresholds'].get('balanced', 0.5)
    
    result['is_alert'] = result['risk_score'] >= threshold
    
    # Статистика
    print(f"\n📊 Результат:")
    print(f"   Позиций: {len(result):,}")
    print(f"   Threshold: {threshold:.2f}")
    print(f"   Алертов: {result['is_alert'].sum():,}")
    
    # Top-K
    if top_k:
        result = result.nsmallest(top_k, 'risk_rank')
        print(f"   Возвращаем Top-{top_k}")
    
    return result


# ================================================================
# ЗАПУСК INFERENCE
# ================================================================

if __name__ == "__main__":
    
    # 1. Загружаем модель
    print("="*60)
    print("🚀 ЗАГРУЗКА МОДЕЛИ")
    print("="*60)
    
    model_data = load_saved_model(
        model_dir="models",
        model_filename="catboost_oos_20260224_111000"  # ← твоё имя файла
    )
    
    # 2. Загружаем СЫРЫЕ данные (минимум 30 дней!)
    print("\n" + "="*60)
    print("📥 ЗАГРУЗКА ДАННЫХ")
    print("="*60)
    
    # Замени на свой путь к данным
    raw_df = final_df.copy()
    # или: raw_df = pd.read_csv("data/stock_data.csv", parse_dates=['date'])
    
    print(f"Загружено: {len(raw_df):,} строк")
    
    # 3. Подготавливаем данные (создаём фичи БЕЗ обрезки)
    df_features = prepare_data_for_inference(
        raw_df,
        date_col='date',
        product_col='code_kag',
        category_col='NM_F',  # или None если нет
        mnn_col=None,         # или 'NM_DT' если есть
    )
    
    # 4. Берём последний день для предсказания
    latest_date = df_features['date'].max()
    df_today = df_features[df_features['date'] == latest_date].copy()
    
    print(f"\n📅 Дата предсказания: {latest_date.date()}")
    print(f"   Позиций: {len(df_today):,}")
    
    # 5. Предсказываем
    print("\n" + "="*60)
    print("🎯 ПРЕДСКАЗАНИЕ")
    print("="*60)
    
    predictions = predict_oos_risk(
        df_today,
        model_data,
        threshold=0.7,
        top_k=200,
    )
    
    # 6. Смотрим результат
    print("\n" + "="*60)
    print("📋 ТОП-20 РИСКОВЫХ ПОЗИЦИЙ")
    print("="*60)
    
    display_cols = ['code_kag', 'risk_score', 'risk_rank', 'is_alert']
    # Добавляем дополнительные колонки если есть
    for col in ['total_stock', 'n_zero', 'days_of_stock', 'NM_F']:
        if col in predictions.columns:
            display_cols.append(col)
    
    print(predictions[display_cols].head(20).to_string())
    
    # 7. Сохраняем отчёт
    output_path = f"reports/oos_alerts_{latest_date.date()}.xlsx"
    os.makedirs("reports", exist_ok=True)
    predictions.to_excel(output_path, index=False)
    print(f"\n💾 Отчёт сохранён: {output_path}")

🚀 ЗАГРУЗКА МОДЕЛИ
✅ Модель загружена: models\catboost_oos_20260224_111000.cbm
   Фичей: 156
   PR-AUC: 0.6329

📥 ЗАГРУЗКА ДАННЫХ
Загружено: 410,581 строк
📊 ПОДГОТОВКА ДАННЫХ ДЛЯ INFERENCE

📅 Период: 2025-12-15 — 2026-03-03 (79 дней)
   Продуктов: 8,937
   Строк: 410,581

🔧 Шаг 1: Создание базовых фичей...


C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\956614280.py:212: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['had_oos_30d'] = (df['oos_count_30d'] > 0).astype(int)
C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\956614280.py:213: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['oos_rate_30d'] = df['oos_count_30d'] / 30
C:\Users\SIMukovoz\AppData\Local\Temp\ipykernel_98744\956614280.py:215: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

   ✅ Колонок: 131

🔧 Шаг 2: Создание предиктивных фичей...
   ✅ Колонок: 171

✅ Готово: 410,581 строк, 171 колонок

📅 Дата предсказания: 2026-03-03
   Позиций: 8,571

🎯 ПРЕДСКАЗАНИЕ
⚠️ Отсутствуют 8 фичей — добавляем как -999:
   ['cat_oos_rate_today', 'cat_oos_rate_7d', 'category_stress', 'cat_n_oos_today', 'cat_oos_ratio', 'cat_oos_growth', 'category_deteriorating', 'is_defect']

📊 Результат:
   Позиций: 8,571
   Threshold: 0.70
   Алертов: 299
   Возвращаем Top-200

📋 ТОП-20 РИСКОВЫХ ПОЗИЦИЙ
        code_kag  risk_score  risk_rank  is_alert  total_stock  n_zero  days_of_stock
406803     23650    0.948281          1      True         5943       4       4.514801
403810      9188    0.944370          2      True         6055       4       8.558317
406044     21147    0.940756          3      True           45       4      16.311622
406457     20368    0.932905          4      True          111       4       8.166949
407735     20330    0.931439          5      True            7       4

In [21]:
predictions

,date,code_kag,puls_stock,gk_stock,katren_stock,protek_stock,farm_stock,МНН,Форма выпуска,Лекарственная форма,...,is_month_start,is_month_end,quarter,quarter_sin,quarter_cos,n_red_flags,critical_combination,risk_score,risk_rank,is_alert
406803,2026-03-03,23650,0,5943,0,0,0,Габапентин,капсулы,табл./драже/капс./пастил.,...,1,0,1,1.0,6.123234e-17,2,0,0.948281,1,True
403810,2026-03-03,9188,0,6055,0,0,0,Метоклопрамид,таблетки,табл./драже/капс./пастил.,...,1,0,1,1.0,6.123234e-17,1,0,0.944370,2,True
406044,2026-03-03,21147,0,45,0,0,0,Калия сульфат+магния сульфат+натрия сульфат,концентрат для приготовления раствора для прие...,р-р внутр.,...,1,0,1,1.0,6.123234e-17,2,0,0.940756,3,True
406457,2026-03-03,20368,0,111,0,0,0,Кетопрофен,гель для наружного применения,мазь/крем/гель,...,1,0,1,1.0,6.123234e-17,1,0,0.932905,4,True
407735,2026-03-03,20330,0,7,0,0,0,Соталол,таблетки,табл./драже/капс./пастил.,...,1,0,1,1.0,6.123234e-17,3,1,0.931439,5,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409281,2026-03-03,29482,0,30,0,0,0,Фрамицетин,спрей назальный,аэрозоль/спрей,...,1,0,1,1.0,6.123234e-17,1,0,0.761794,196,True
406751,2026-03-03,23160,2166,2826,3216,0,0,Ацетилсалициловая кислота+кофеин+парацетамол,таблетки,табл./драже/капс./пастил.,...,1,0,1,1.0,6.123234e-17,3,0,0.761230,197,True
408240,2026-03-03,30559,0,85,0,0,0,NaN,NaN,БАД,...,1,0,1,1.0,6.123234e-17,1,0,0.760876,198,True
405489,2026-03-03,30891,0,1,0,0,0,NaN,NaN,БАД,...,1,0,1,1.0,6.123234e-17,3,0,0.759940,199,True
